In [16]:
!pip install -q transformers datasets torch evaluate accelerate scikit-learn

In [17]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True Tesla T4


In [18]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
import evaluate
from transformers import AutoTokenizer,AutoModelForSequenceClassification, TrainingArguments,Trainer
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
import torch
import os
from transformers import EarlyStoppingCallback, DataCollatorWithPadding, TrainingArguments
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

In [19]:
# Loading the small labelled Cyber Threat Intelligence V2 dataset
dataset = load_dataset("Olec/cyber-threat-intelligence_v2")

In [20]:
# Creating a method to add entity markers around the head and tail entities in the text sentence
def insert_entity_markers(text, head_start, head_end, tail_start, tail_end):
  spans = sorted([(head_start, head_end, "[E1]", "[/E1]"), #adding tags for the start and end of the entity
                  (tail_start, tail_end, "[E2]", "[/E2]")],
                 key=lambda s: -s[0]) #sorting backwards so inserting tags doesn't ruin other indices
  out = text
    # Inserting tags from right to left
  for start, end, open_tag, close_tag in spans:
      out = out[:end] + f" {close_tag}" + out[end:] # Inserting closing tag
      out = out[:start] + f"{open_tag} " + out[start:] # Inserting opening tag
  return out

In [21]:
# Method for converting the dataset into a relation-level DataFrame with marked entities
def explode_all_splits(dataset):
    rows = []
    # looping through each split in the dataset
    for split_name in dataset.keys():
        # looping through each individual text sample in the current split
        for sample in dataset[split_name]:
            text = sample["text"]
            # Creating a lookup dictionary for entities using their IDs
            entity_map = {e["id"]: e for e in sample["entities"]}
            # processing each relation pair in the sample
            for rel in sample["relations"]:
                # Gettting the head and tail entities for the current relation
                head = entity_map.get(rel["from_id"])
                tail = entity_map.get(rel["to_id"])
                if head is None or tail is None:
                    continue  # Skipping if either entity is missing
                # Adding entity markers to the text
                marked_text = insert_entity_markers(text,head["start_offset"], head["end_offset"],tail["start_offset"], tail["end_offset"],)
                # Storing the processed relation as a new row
                rows.append({ "marked_text": marked_text,"head_type": head["label"], "tail_type": tail["label"],"relation": rel["type"],})
    return pd.DataFrame(rows)

In [22]:
# Converting all dataset splits into a single DataFrame
all_df = explode_all_splits(dataset)
#total number of relation examples
print(f"Total relation examples across all original splits: {len(all_df)}")
all_df["relation"].value_counts()

Total relation examples across all original splits: 1095


,count
relation,
uses,192
duplicate-of,181
indicates,156
targets,145
related-to,96
located-at,76
has,62
exploits,35
attributed-to,23


In [23]:
# Splitting the dataset into train, validation and test sets
def stratified_split(df, label_col="relation", train_frac=0.7, val_frac=0.15, seed=42):
     # Setting the random seed for reproducibility
    rng = np.random.RandomState(seed)
    train_idx, val_idx, test_idx = [], [], []
    # Grouping the data by relation label
    for label, group in df.groupby(label_col):
        # Shuffling the indices for the current label
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        # If there are too few samples,we place them in the training set
        if n < 3:
            train_idx.extend(idx)
            continue
        # Calculating the number of samples for each split
        n_train = max(1, int(round(n * train_frac)))
        # To make sure atleast one sample remains for validation and test sets
        n_val = max(1, int(round(n * val_frac)))
        n_train = min(n_train, n - 2)
        # Assigning indices to the train, validation, and test sets
        train_idx.extend(idx[:n_train])
        val_idx.extend(idx[n_train:n_train + n_val])
        test_idx.extend(idx[n_train + n_val:])
 # Returning the three splits as separate DataFrames
    return (
        df.loc[train_idx].reset_index(drop=True),
        df.loc[val_idx].reset_index(drop=True),
        df.loc[test_idx].reset_index(drop=True),
    )

In [24]:
#  Splitting the dataset into training, validation, and test sets and printing the number of samples each split
train_df, valid_df, test_df = stratified_split(all_df)
print(f"train={len(train_df)}  valid={len(valid_df)}  test={len(test_df)}")

train=766  valid=163  test=166


In [25]:
# Verify that all relation types in the validation and test sets are also present in the training set
train_labels = set(train_df["relation"])
# Finding relation types that are missing from the training set
unseen_in_val = set(valid_df["relation"]) - train_labels
unseen_in_test = set(test_df["relation"]) - train_labels
print("Relation types in val but not train:", unseen_in_val)
print("Relation types in test but not train:", unseen_in_test)

# Stoping the execution if unseen relation types are found
assert not unseen_in_val and not unseen_in_test, "Still have unseen labels - stratified_split needs a larger min-count threshold"
print("Every eval relation type is covered in training.")

Relation types in val but not train: set()
Relation types in test but not train: set()
Every eval relation type is covered in training.


In [26]:
# Encoding the relation labels into numerical values and learning the mapping from relation names to numeric labels
encoder = LabelEncoder()
encoder.fit(all_df["relation"])

# Applying the label encoding to each dataset split
train_df["label"] = encoder.transform(train_df["relation"])
valid_df["label"] = encoder.transform(valid_df["relation"])
test_df["label"] = encoder.transform(test_df["relation"])

num_labels = len(encoder.classes_)
print(f"{num_labels} relation classes: {list(encoder.classes_)}")

25 relation classes: ['attributed-to', 'authored-by', 'based-on', 'beacons-to', 'communicates-with', 'compromises', 'consists-of', 'controls', 'delivers', 'downloads', 'drops', 'duplicate-of', 'exfiltrates-to', 'exploits', 'has', 'hosts', 'impersonates', 'indicates', 'located-at', 'originates-from', 'owns', 'related-to', 'targets', 'uses', 'variant-of']


In [27]:
# Defining two pretrained models BERT and SecureBERT to be used for relation extraction

MODELS = {
    "BERT": "bert-base-uncased",
    "SecureBERT": "ehsanaghaei/SecureBERT"
}


# Setting the maximum token length for model inputs
MAX_LENGTH = 160

In [28]:
from sklearn.utils.class_weight import compute_class_weight
import torch

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_labels),
    y=train_df["label"].to_numpy(),
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

tensor([ 1.9150,  1.9150, 30.6400,  4.3771,  3.4044, 30.6400,  4.3771, 15.3200,
         3.4044,  2.5533, 15.3200,  0.2413, 15.3200,  1.2767,  0.7126,  5.1067,
         7.6600,  0.2811,  0.5781, 15.3200, 30.6400,  0.4573,  0.3004,  0.2287,
         3.0640])


In [29]:
# Main Training Loop for Both Models
# Define the metrics function  to calculate evaluation metrics for model performance
def compute_metrics(eval_pred):
     # Extracting model predictions and true labels
    logits, labels = eval_pred
    # Converting model outputs into predicted class labels
    preds = np.argmax(logits, axis=-1)
     # Calculating accuracy score
    acc = accuracy_score(labels, preds)
    # Calculating macro precision, recall and F1 score
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    # Calculating weighted precision, recall, and F1 score
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1,
    }

# Storing final evaluation results for each model for comparison
all_final_metrics = {}

In [30]:
for model_key, model_path in MODELS.items():
    print("\n" + "="*50)
    print(f"STARTING PIPELINE FOR: {model_key} ({model_path})")
    print("="*50 + "\n")

    # Initializing Tokenizer for a specific model
    print(f"[{model_key}] Loading tokenizer")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    # Adding entity marker tokens used for relation extraction
    tokenizer.add_special_tokens({"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]})

    # Converting pandas DataFrames into Hugging Face datasets and tokenize the text
    def to_hf_dataset(df):
         # Creating Hugging Face dataset with input text and labels
        ds = Dataset.from_pandas(df[["marked_text", "label"]])
        # Tokenizing the text and apply maximum sequence length limit
        ds = ds.map( lambda batch: tokenizer(batch["marked_text"], truncation=True, max_length=MAX_LENGTH),
                     batched=True,desc=f"Tokenizing for {model_key}")
        # Renaming label column to match Hugging Face Trainer requirements
        ds = ds.rename_column("label", "labels")
        # Removing original text column after tokenization
        ds = ds.remove_columns(["marked_text"])
        return ds

    print(f"[{model_key}] Tokenizing datasets")
    train_ds = to_hf_dataset(train_df)
    valid_ds = to_hf_dataset(valid_df)
    test_ds = to_hf_dataset(test_df)

    # Creating a data collator for dynamic padding during training
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Initializing the Model and resizng embeddings for the special tokens
    print(f"[{model_key}] Initializing the model")
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=num_labels)
    # Resizing model embeddings to include the added entity marker tokens
    model.resize_token_embeddings(len(tokenizer))

    # Defining separate folders for results and saved models
    output_dir = f"./{model_key.lower()}_results"
    final_save_dir = f"{model_key.lower()}_baseline_fixed"

    # training parameters
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        label_smoothing_factor=0.1,
        num_train_epochs=30,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    # Initializing Hugging Face Trainer with evaluation metrics and early stopping
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )

    # Training the model on the training dataset
    print(f"[{model_key}] Starting the training of the model")
    trainer.train()

    # Evaluating on test set
    print(f"[{model_key}] Evaluating on test set")
    test_metrics = trainer.evaluate(test_ds)
    # Storing the final metrics for comparing different models
    all_final_metrics[model_key] = test_metrics

    print(f"\nTest set metrics for {model_key}:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v}")

    # Generating Classification Report
    preds_output = trainer.predict(test_ds)
    # Converting model outputs into predicted labels
    preds = np.argmax(preds_output.predictions, axis=-1)
    labels = preds_output.label_ids
    # Getting only labels present in the test dataset
    present_labels = np.unique(labels)
    present_names = encoder.classes_[present_labels]

    # Displaying the classification report with precision, recall and F1-score
    print(f"\nClassification Report for {model_key}:")
    print(classification_report(
        labels, preds,
        labels=present_labels,
        target_names=present_names,
        zero_division=0,
    ))

    print(f"[{model_key}] Saving model and tokenizer to {final_save_dir}")
    trainer.save_model(final_save_dir)
    tokenizer.save_pretrained(final_save_dir)

    # Cleaning up GPU memory before next model loop
    del model
    del trainer
    torch.cuda.empty_cache()


STARTING PIPELINE FOR: BERT (bert-base-uncased)

[BERT] Loading tokenizer
[BERT] Tokenizing datasets


Tokenizing for BERT:   0%|          | 0/766 [00:00<?, ? examples/s]

Tokenizing for BERT:   0%|          | 0/163 [00:00<?, ? examples/s]

Tokenizing for BERT:   0%|          | 0/166 [00:00<?, ? examples/s]

[BERT] Initializing the model


...

[SecureBERT] Evaluating on test set


Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
0.140530,1.803473,24,0.759036,0.531928,0.553955,0.537297,0.758126,0.759036,0.752753



Test set metrics for SecureBERT:
  eval_loss: 1.803472638130188
  eval_accuracy: 0.7590361445783133
  eval_macro_precision: 0.531928188770294
  eval_macro_recall: 0.5539547875344264
  eval_macro_f1: 0.5372973023235433
  eval_weighted_precision: 0.7581257233571755
  eval_weighted_recall: 0.7590361445783133
  eval_weighted_f1: 0.7527526144509346



Classification Report for SecureBERT:
                   precision    recall  f1-score   support

    attributed-to       0.75      0.75      0.75         4
      authored-by       0.67      0.50      0.57         4
       beacons-to       1.00      1.00      1.00         1
communicates-with       0.67      1.00      0.80         2
      consists-of       0.00      0.00      0.00         1
         delivers       0.50      0.50      0.50         2
        downloads       0.25      0.50      0.33         2
            drops       0.00      0.00      0.00         1
     duplicate-of       0.70      0.85      0.77        27
   exfiltrates-to       0.00      0.00      0.00         1
         exploits       0.83      0.83      0.83         6
              has       0.80      0.80      0.80        10
            hosts       0.00      0.00      0.00         1
     impersonates       1.00      1.00      1.00         1
        indicates       0.95      0.75      0.84        24
       located-a

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [31]:
print("\nFinal Comparison\n")

# Creating a list to store evaluation results for each model
comparison = []
# Extracting the relevant metrics from each model's evaluation results
for model_name, metrics in all_final_metrics.items():
    # Storing each model's performance metrics
    comparison.append({
        "Model": model_name,
        "Accuracy": metrics["eval_accuracy"],
        "Macro F1": metrics["eval_macro_f1"],
        "Weighted F1": metrics["eval_weighted_f1"],
        "Macro Precision": metrics["eval_macro_precision"],
        "Macro Recall": metrics["eval_macro_recall"],
    })

# Comparing and printing the results
comparison_df = pd.DataFrame(comparison)
print(comparison_df)


Final Comparison

        Model  Accuracy  Macro F1  Weighted F1  Macro Precision  Macro Recall
0        BERT  0.728916  0.518186     0.726536         0.499393      0.569372
1  SecureBERT  0.759036  0.537297     0.752753         0.531928      0.553955
